# 7.5 MUSDB18-HQ 客观评估

本 notebook 使用 MUSDB18-HQ 的 reference stems 计算两类客观指标：**SDR（BSS Eval）** 作为主指标，**SI-SDR** 作为尺度不变的对照指标。它读取 `07_4` 已保存的分离结果，不重新运行模型；同一张表中保留曲目名称、case_id、模型 ID、模型显示名、输出模式、评估任务和 stem 映射。

SDR 是音乐源分离领域的标准评估口径，由 `museval`（BSSEval v4）实现，也是 MUSDB18 榜单与 MDX/SDX 挑战赛的排名指标；它先用最优线性失真滤波器把估计对齐到参考，再衡量残差。SI-SDR 只校正全局尺度，是语音分离的标准指标，这里并列报告用于对照。两者数值通常接近，但一般 SDR 略高，因为线性滤波匹配能吸收更多失真。

MUSDB18-HQ 的标准任务是 `vocals/drums/bass/other`。2-stem 模型只能进入 `vocals/accompaniment` 任务；6-stem 模型的 `other/guitar/piano` 会合并后再和 MUSDB 的 `other` 比较。


## 1. 路径与运行参数

本单元设置输出目录、评估采样率、默认评估曲目数和片段长度。若 `07_4_input_cases.csv` 中已有 MUSDB 案例，优先沿用这些案例，保证评估曲目与预训练模型输出一致。


In [ ]:
import csv
from dataclasses import dataclass, replace
import os
import re
import sys
import tempfile
import textwrap
from pathlib import Path

# matplotlib/numba 缓存目录：用跨平台的系统临时目录
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "mplconfig"))
os.environ.setdefault("NUMBA_CACHE_DIR", str(Path(tempfile.gettempdir()) / "numba_cache"))

# 路径推断：从 cwd 向上找含 CODE/chapter07/_common 的目录；NOTEBOOK_DIR 指向 CODE/chapter07/
_p = Path.cwd()
while not (_p / "CODE" / "chapter07" / "_common").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/chapter07/_common 的目录），请在项目内运行本 Notebook")
    _p = _parent
NOTEBOOK_DIR = _p / "CODE" / "chapter07"
CODE_ROOT = NOTEBOOK_DIR.parent
REPO_ROOT = CODE_ROOT.parent
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

FIG_DIR = NOTEBOOK_DIR / "output_figures"
FIG_DIR.mkdir(exist_ok=True)
TABLE_DIR = NOTEBOOK_DIR / "outputs" / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
OUT_AUDIO_DIR = NOTEBOOK_DIR / "output_audio" / "07_4"

EVAL_SR = int(os.environ.get("CHAPTER07_EVAL_SR", "22050"))
DEFAULT_EVAL_DURATION = float(os.environ.get("CHAPTER07_EVAL_DURATION", "20"))
MAX_EVAL_TRACKS = int(os.environ.get("CHAPTER07_MAX_EVAL_TRACKS", "2"))

print("NOTEBOOK_DIR:", NOTEBOOK_DIR.relative_to(REPO_ROOT))
print("EVAL_SR:", EVAL_SR)
print("DEFAULT_EVAL_DURATION:", DEFAULT_EVAL_DURATION)
print("MAX_EVAL_TRACKS:", MAX_EVAL_TRACKS)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from chapter07._common.audio_io import load_audio
from chapter07._common.dataset_paths import find_musdb_root, list_musdb_tracks, load_musdb_track
from chapter07._common.evaluation import (
    map_estimates_to_musdb_tasks,
    mean_sdr_by_model_stem,
    mean_sisdr_by_model_stem,
    mixture_as_estimates,
    musdb_reference_tasks,
    oracle_irm_estimates,
    sisdr_rows,
    write_metric_rows,
)
from chapter07._common.external_diagnostics import (
    collect_manifest_output_paths,
    discover_output_audio_paths,
    output_mode_from_stems,
    repo_relative_path,
    resolve_repo_path,
)
from chapter07._common.plotting import BAR_GRAY, FIGURE_SAVE_DPI, display_label, setup_plot_style
from chapter07._common.synthesis import make_synthetic_mixture


## 2. 表格工具

这些函数用于读取 `07_4` 的案例表、模型表与输出 manifest，并把本节产生的评估曲目、模型覆盖和指标汇总写成 CSV。


In [ ]:
MetricValue = str | float | int
TableRow = dict[str, MetricValue]
AudioByStem = dict[str, np.ndarray]


@dataclass
class EvalTrack:
    case_id: str
    track: str
    source_path: Path | None
    track_dir: Path | None
    start_sec: float
    duration_sec: float
    reference_type: str
    data: AudioByStem | None = None


def read_csv_rows(path: Path) -> list[dict[str, str]]:
    path = Path(path)
    if not path.exists():
        return []
    with path.open("r", newline="", encoding="utf-8") as f:
        return [{key: value or "" for key, value in row.items()} for row in csv.DictReader(f)]


def write_rows(path: Path, rows: list[TableRow], fieldnames: list[str] | None = None) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if fieldnames is None:
        fieldnames = []
        for row in rows:
            for key in row:
                if key not in fieldnames:
                    fieldnames.append(key)
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def print_rows(rows: list[TableRow], columns: list[str], limit: int = 20) -> None:
    if not rows:
        print("(no rows)")
        return
    shown = rows[:limit]
    widths = {
        col: min(max(len(str(row.get(col, ""))) for row in shown + [{col: col}]), 44)
        for col in columns
    }
    print(" | ".join(col.ljust(widths[col]) for col in columns))
    print("-+-".join("-" * widths[col] for col in columns))
    for row in shown:
        cells = []
        for col in columns:
            text = str(row.get(col, ""))
            if len(text) > widths[col]:
                text = text[: widths[col] - 3] + "..."
            cells.append(text.ljust(widths[col]))
        print(" | ".join(cells))
    if len(rows) > limit:
        print(f"... {len(rows) - limit} more rows")


def slugify(value: str) -> str:
    clean = re.sub(r"[^0-9A-Za-z]+", "_", value).strip("_").lower()
    return clean or "track"


def compact_label(value: object, width: int = 76) -> str:
    return textwrap.shorten(" ".join(str(value).split()), width=width, placeholder="...")


## 3. 载入 MUSDB 评估曲目

优先使用 `07_4_input_cases.csv` 中已经分离过的 MUSDB 案例；若该表不存在，再扫描本地 MUSDB18-HQ。找不到 MUSDB18-HQ 时使用合成样例，只验证 SDR/SI-SDR 与绘图流程。


In [ ]:
MUSDB_STEMS = ("vocals", "drums", "bass", "other")


def synthetic_eval_track() -> EvalTrack:
    sources = make_synthetic_mixture(sr=EVAL_SR, duration=DEFAULT_EVAL_DURATION, seed=55)
    zeros = np.zeros_like(sources["mixture"])
    data: AudioByStem = {
        "mixture": sources["mixture"],
        "vocals": sources["harmonic"],
        "drums": sources["percussive"],
        "bass": sources["bass"],
        "other": zeros,
    }
    return EvalTrack(
        case_id="synthetic_eval",
        track="synthetic_eval",
        source_path=None,
        track_dir=None,
        start_sec=0.0,
        duration_sec=DEFAULT_EVAL_DURATION,
        reference_type="synthetic",
        data=data,
    )


def candidate_tracks_from_07_4() -> list[EvalTrack]:
    rows = read_csv_rows(TABLE_DIR / "07_4_input_cases.csv")
    candidates: list[EvalTrack] = []
    for row in rows:
        if row.get("family") != "musdb18hq":
            continue
        source_path = resolve_repo_path(row.get("source", ""), REPO_ROOT)
        track_dir = source_path.parent
        if not track_dir.exists():
            continue
        candidates.append(
            EvalTrack(
                case_id=row.get("case_id", "").strip() or f"musdb_{slugify(track_dir.name)}",
                track=row.get("display_name", "").strip() or track_dir.name,
                source_path=source_path,
                track_dir=track_dir,
                start_sec=float(row.get("start_sec") or 0.0),
                duration_sec=float(row.get("duration_sec") or DEFAULT_EVAL_DURATION),
                reference_type="musdb18hq_from_07_4",
            )
        )
    return candidates[:MAX_EVAL_TRACKS]


def candidate_tracks_from_musdb_root() -> list[EvalTrack]:
    root = find_musdb_root()
    if root is None:
        return []
    candidates: list[EvalTrack] = []
    for split in ("test", "valid", "train"):
        tracks = list_musdb_tracks(root, split=split)
        if not tracks:
            continue
        for track_dir in tracks[:MAX_EVAL_TRACKS]:
            candidates.append(
                EvalTrack(
                    case_id=f"musdb_{slugify(track_dir.name)}",
                    track=track_dir.name,
                    source_path=track_dir / "mixture.wav",
                    track_dir=track_dir,
                    start_sec=0.0,
                    duration_sec=DEFAULT_EVAL_DURATION,
                    reference_type=f"musdb18hq_{split}",
                )
            )
        break
    return candidates


def load_eval_track(candidate: EvalTrack) -> EvalTrack | None:
    if candidate.reference_type == "synthetic":
        return candidate
    if candidate.track_dir is None:
        return None
    track_dir = candidate.track_dir
    data = load_musdb_track(
        track_dir,
        duration=candidate.duration_sec,
        sr=EVAL_SR,
        start=candidate.start_sec,
        mono=True,
    )
    if not {"mixture", *MUSDB_STEMS}.issubset(data):
        print("跳过不完整 MUSDB 曲目:", track_dir.name)
        return None
    return replace(candidate, data=data)


track_candidates = candidate_tracks_from_07_4()
if not track_candidates:
    track_candidates = candidate_tracks_from_musdb_root()

eval_tracks = [track for track in (load_eval_track(item) for item in track_candidates) if track is not None]
if not eval_tracks:
    print("未找到可用 MUSDB18-HQ reference；使用合成样例验证评估流程。")
    eval_tracks = [synthetic_eval_track()]

track_rows: list[TableRow] = []
for track in eval_tracks:
    source_path = track.source_path
    track_rows.append(
        {
            "case_id": track.case_id,
            "track": track.track,
            "reference_type": track.reference_type,
            "source_path": repo_relative_path(source_path, REPO_ROOT) if source_path else "",
            "start_sec": track.start_sec,
            "duration_sec": track.duration_sec,
            "reference_stems": ";".join(MUSDB_STEMS),
        }
    )

write_rows(TABLE_DIR / "07_5_evaluation_tracks.csv", track_rows)
print_rows(track_rows, ["case_id", "track", "reference_type", "source_path", "duration_sec"])


## 4. Baseline 与 oracle mask

`mixture_baseline` 把混音直接当作每个目标的估计；`oracle_irm_mixture_phase` 使用 reference magnitude 生成 ideal ratio mask，但仍复用 mixture phase。前者是下限参照，后者是 mask 与相位约束下的上限。


In [ ]:
def metric_context(track: EvalTrack, display_name: str, mode: str, task: str) -> dict[str, MetricValue]:
    return {
        "case_id": track.case_id,
        "model_display_name": display_name,
        "mode": mode,
        "task": task,
        "reference_type": track.reference_type,
        "duration_sec": track.duration_sec,
    }


rows: list[dict[str, MetricValue]] = []
for track in eval_tracks:
    data = track.data
    if data is None:
        continue
    mixture = data["mixture"]
    reference_tasks = musdb_reference_tasks({stem: data[stem] for stem in MUSDB_STEMS})
    for task, references in reference_tasks.items():
        stems = tuple(references)
        mixture_estimates = mixture_as_estimates(mixture, stems=stems)
        rows.extend(
            sisdr_rows(
                track.track,
                "mixture_baseline",
                references,
                mixture_estimates,
                extra=metric_context(track, "Mixture baseline", task, task),
                estimate_sources={stem: ("mixture",) for stem in stems},
            )
        )

        oracle_estimates = oracle_irm_estimates(
            mixture,
            references,
            stems=stems,
            n_fft=2048,
            hop_length=512,
        )
        rows.extend(
            sisdr_rows(
                track.track,
                "oracle_irm_mixture_phase",
                references,
                oracle_estimates,
                extra=metric_context(
                    track,
                    "Oracle IRM (mixture phase)",
                    task,
                    task,
                ),
                estimate_sources={stem: (stem,) for stem in stems},
            )
        )

print("metric rows after baselines:", len(rows))


## 5. 读取 07_4 预训练模型输出

本单元读取 `07_4_output_audio_manifest.csv`，自动纳入 `07_4` 已经生成的全部 MUSDB 模型输出。若 manifest 不存在，则扫描 `output_audio/07_4/<case_id>/<model_id>/`。


In [ ]:
def model_display_lookup() -> dict[str, str]:
    lookup = {
        "mixture_baseline": "Mixture baseline",
        "oracle_irm_mixture_phase": "Oracle IRM (mixture phase)",
    }
    for row in read_csv_rows(TABLE_DIR / "07_4_model_setup.csv"):
        model_id = row.get("model_id", "").strip()
        if model_id:
            lookup[model_id] = row.get("display_name", "").strip() or model_id
    for row in read_csv_rows(TABLE_DIR / "07_4_reference_optional_models.csv"):
        model_id = row.get("model_id", "").strip()
        if model_id and model_id not in lookup:
            lookup[model_id] = model_id
    return lookup


def model_setup_rows() -> list[dict[str, str]]:
    return read_csv_rows(TABLE_DIR / "07_4_model_setup.csv")


def infer_model_mode(model_id: str, stems: list[str] | tuple[str, ...] | None = None) -> str:
    if stems:
        return output_mode_from_stems(stems)
    clean = model_id.lower()
    if "6stem" in clean or "6s" in clean:
        return "6stems"
    if "2stem" in clean or "roformer" in clean:
        return "2stems"
    if "4stem" in clean or "openunmix" in clean or "hdemucs" in clean:
        return "4stems"
    return ""


def coverage_status(setup_row: dict[str, str], stems: list[str]) -> str:
    if stems:
        return "outputs_found"
    if setup_row.get("model_ready") == "False":
        return "model_not_ready_in_07_4"
    return "no_07_4_output_for_case"


case_to_track = {track.case_id: track.track for track in eval_tracks}
case_ids = set(case_to_track)
model_names = model_display_lookup()

output_manifest_path = TABLE_DIR / "07_4_output_audio_manifest.csv"
model_output_paths = collect_manifest_output_paths(output_manifest_path, REPO_ROOT, case_ids=case_ids)
output_source = "07_4_output_audio_manifest.csv"
if not model_output_paths:
    model_output_paths = discover_output_audio_paths(OUT_AUDIO_DIR, case_ids=case_ids)
    output_source = "output_audio/07_4 directory scan"

inventory_rows: list[TableRow] = []
for case_id, models in sorted(model_output_paths.items()):
    for model_id, stem_paths in sorted(models.items()):
        stems = sorted(stem_paths)
        inventory_rows.append(
            {
                "case_id": case_id,
                "track": case_to_track.get(case_id, case_id),
                "model_id": model_id,
                "model_display_name": model_names.get(model_id, model_id),
                "mode": output_mode_from_stems(stems),
                "stem_count": len(stems),
                "stems": ";".join(stems),
                "source": output_source,
            }
        )

coverage_rows: list[TableRow] = []
setup_rows = model_setup_rows()
setup_model_ids = [row.get("model_id", "").strip() for row in setup_rows if row.get("model_id", "").strip()]
observed_model_ids = sorted({str(row["model_id"]) for row in inventory_rows})
for model_id in sorted(set(setup_model_ids) | set(observed_model_ids)):
    setup_row = next((row for row in setup_rows if row.get("model_id") == model_id), {})
    for case_id in sorted(case_ids):
        stems = sorted(model_output_paths.get(case_id, {}).get(model_id, {}))
        coverage_rows.append(
            {
                "case_id": case_id,
                "track": case_to_track.get(case_id, case_id),
                "model_id": model_id,
                "model_display_name": model_names.get(model_id, model_id),
                "mode": infer_model_mode(model_id, stems),
                "dependency_ready": setup_row.get("dependency_ready", ""),
                "model_ready": setup_row.get("model_ready", ""),
                "status": coverage_status(setup_row, stems),
                "stems": ";".join(stems),
                "notes": setup_row.get("notes", ""),
            }
        )

write_rows(TABLE_DIR / "07_5_model_inventory.csv", inventory_rows)
write_rows(TABLE_DIR / "07_5_model_coverage.csv", coverage_rows)
print("model output source:", output_source)
print_rows(inventory_rows, ["case_id", "track", "model_id", "mode", "stems"], limit=24)
print()
print_rows(coverage_rows, ["case_id", "model_id", "mode", "dependency_ready", "model_ready", "status", "stems"], limit=24)


## 6. 计算预训练模型 SDR 与 SI-SDR

4-stem 输出直接和 MUSDB 四轨比较；2-stem 输出只比较 `vocals/accompaniment`；6-stem 输出把 `other/guitar/piano` 合并为 MUSDB 的 `other`。`estimate_stems` 字段记录每个评估目标由哪些模型输出 stem 组成。每个 stem 同时计算 SDR（BSS Eval）与 SI-SDR。


In [ ]:
def load_model_estimates(stem_paths: dict[str, Path], duration: float) -> dict[str, np.ndarray]:
    estimates: dict[str, np.ndarray] = {}
    for stem, path in sorted(stem_paths.items()):
        audio, _ = load_audio(path, sr=EVAL_SR, mono=True, duration=duration)
        estimates[stem] = audio
    return estimates


for track in eval_tracks:
    if track.reference_type == "synthetic" or track.data is None:
        continue
    data = track.data
    reference_tasks = musdb_reference_tasks({stem: data[stem] for stem in MUSDB_STEMS})
    case_id = track.case_id
    for model_id, stem_paths in sorted(model_output_paths.get(case_id, {}).items()):
        estimates = load_model_estimates(stem_paths, track.duration_sec)
        mapped_tasks = map_estimates_to_musdb_tasks(estimates)
        mode = output_mode_from_stems(tuple(stem_paths))
        for task, references in reference_tasks.items():
            if task not in mapped_tasks:
                continue
            mapped_estimates, estimate_sources = mapped_tasks[task]
            rows.extend(
                sisdr_rows(
                    track.track,
                    model_id,
                    references,
                    mapped_estimates,
                    extra=metric_context(
                        track,
                        model_names.get(model_id, model_id),
                        mode,
                        task,
                    ),
                    estimate_sources=estimate_sources,
                )
            )

print("total metric rows:", len(rows))


## 7. 保存指标与汇总表

明细表保留每首曲目、每个模型、每个 stem 的 SDR 与 SI-SDR。汇总表按 `task/model/stem` 与 `task/model` 聚合，避免把 2-stem、4-stem、6-stem 的任务定义混在一起排名。汇总同时给出均值与中位数；跨曲目比较时以中位数为准，对个别异常曲目更稳健，这也是 `museval` 的默认聚合方式。


In [ ]:
def finite_values(values: list[float]) -> list[float]:
    return [value for value in values if np.isfinite(value)]


def summarize_by_fields(rows: list[dict[str, MetricValue]], fields: list[str]) -> list[TableRow]:
    grouped: dict[tuple[str, ...], dict[str, list[float]]] = {}
    for row in rows:
        key = tuple(str(row.get(field, "")) for field in fields)
        bucket = grouped.setdefault(key, {"sdr": [], "si_sdr": []})
        for metric in ("sdr", "si_sdr"):
            if metric in row:
                value = float(row[metric])
                if np.isfinite(value):
                    bucket[metric].append(value)
    summary: list[TableRow] = []
    for key, bucket in grouped.items():
        item: TableRow = {field: key[index] for index, field in enumerate(fields)}
        item["n"] = int(len(bucket["sdr"]) or len(bucket["si_sdr"]))
        for metric in ("sdr", "si_sdr"):
            values = bucket[metric]
            if not values:
                continue
            arr = np.asarray(values, dtype=float)
            item[f"mean_{metric}"] = float(np.mean(arr))
            item[f"median_{metric}"] = float(np.median(arr))
            item[f"min_{metric}"] = float(np.min(arr))
            item[f"max_{metric}"] = float(np.max(arr))
        summary.append(item)
    return sorted(summary, key=lambda row: (str(row.get("task", "")), str(row.get("model", "")), str(row.get("stem", ""))))


metric_path = TABLE_DIR / "07_5_musdb18hq_metrics.csv"
write_metric_rows(metric_path, rows)

stem_summary_rows = summarize_by_fields(
    rows,
    ["task", "model", "model_display_name", "mode", "stem"],
)
model_summary_rows = summarize_by_fields(
    rows,
    ["task", "model", "model_display_name", "mode"],
)
write_rows(TABLE_DIR / "07_5_musdb18hq_summary.csv", stem_summary_rows)
write_rows(TABLE_DIR / "07_5_model_task_summary.csv", model_summary_rows)

print("wrote:", repo_relative_path(metric_path, REPO_ROOT))
print_rows(stem_summary_rows, ["task", "model", "mode", "stem", "n", "median_sdr", "median_si_sdr"], limit=28)
print()
print_rows(model_summary_rows, ["task", "model", "mode", "n", "median_sdr", "median_si_sdr"], limit=28)


## 8. 绘制模型与 stem 指标图

第一张图显示 `task/model/stem` 的中位 SDR，适合定位某个 stem 的薄弱点；第二张图按 `task/model` 聚合，适合快速查看哪些模型输出已经进入评估。两图均以 SDR（BSS Eval）为主指标，与 MUSDB18 / MDX 挑战赛口径一致。


In [ ]:
def plot_stem_summary(rows: list[TableRow]) -> None:
    if not rows:
        fig, ax = plt.subplots(figsize=(8, 3))
        ax.text(0.5, 0.5, "暂无 SDR 汇总数据", ha="center", va="center")
        ax.axis("off")
        fig.savefig(FIG_DIR / "07_5_sdr_by_stem.png", bbox_inches="tight", dpi=FIGURE_SAVE_DPI)
        plt.show()
        return
    plot_rows = sorted(rows, key=lambda row: (str(row["task"]), str(row["model"]), str(row["stem"])))
    labels = [
        compact_label(
            f"{row['task']} | {row['model_display_name']} | {display_label(str(row['stem']))}",
            88,
        )
        for row in plot_rows
    ]
    values = [float(row["median_sdr"]) for row in plot_rows]
    height = max(5.0, 0.32 * len(plot_rows) + 1.8)
    fig, ax = plt.subplots(figsize=(12, height))
    y_pos = np.arange(len(values))
    ax.barh(y_pos, values, color=BAR_GRAY)
    ax.set_yticks(y_pos, labels)
    ax.invert_yaxis()
    ax.set_xlabel("中位 SDR (dB)")
    ax.set_title("MUSDB18-HQ：按任务、模型与 stem 汇总（BSS Eval SDR）")
    ax.grid(axis="x", alpha=0.25)
    ax.grid(axis="y", visible=False)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "07_5_sdr_by_stem.png", bbox_inches="tight", dpi=FIGURE_SAVE_DPI)
    plt.show()


def plot_model_summary(rows: list[TableRow]) -> None:
    if not rows:
        fig, ax = plt.subplots(figsize=(8, 3))
        ax.text(0.5, 0.5, "暂无模型汇总数据", ha="center", va="center")
        ax.axis("off")
        fig.savefig(FIG_DIR / "07_5_model_task_summary.png", bbox_inches="tight", dpi=FIGURE_SAVE_DPI)
        plt.show()
        return
    plot_rows = sorted(rows, key=lambda row: (str(row["task"]), float(row["median_sdr"])))
    labels = [
        compact_label(f"{row['task']} | {row['model_display_name']} ({row['mode']})", 88)
        for row in plot_rows
    ]
    values = [float(row["median_sdr"]) for row in plot_rows]
    height = max(4.5, 0.42 * len(plot_rows) + 1.6)
    fig, ax = plt.subplots(figsize=(11, height))
    y_pos = np.arange(len(values))
    ax.barh(y_pos, values, color="0.35")
    ax.set_yticks(y_pos, labels)
    ax.set_xlabel("中位 SDR (dB)")
    ax.set_title("MUSDB18-HQ：按任务与模型聚合（BSS Eval SDR）")
    ax.grid(axis="x", alpha=0.25)
    ax.grid(axis="y", visible=False)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "07_5_model_task_summary.png", bbox_inches="tight", dpi=FIGURE_SAVE_DPI)
    plt.show()


setup_plot_style()
plot_stem_summary(stem_summary_rows)
plot_model_summary(model_summary_rows)

legacy_summary = mean_sisdr_by_model_stem(rows)
sdr_summary = mean_sdr_by_model_stem(rows)
print("legacy model/stem summary keys:", len(legacy_summary), "sdr keys:", len(sdr_summary))


## 9. 解读要点

- `case_id` 与 `track` 同时保留：前者用于匹配 `07_4` 输出目录，后者用于人工识别曲目。
- `model` 是可复现实验的稳定 ID，`model_display_name` 是读表时使用的模型名称。
- `task` 区分 `musdb_4stem` 与 `vocals_accompaniment_2stem`，不要把两个任务的均值直接混成一个排行榜。
- `estimate_stems` 记录模型输出到评估目标的映射，例如 6-stem 的 `other` 由 `other;guitar;piano` 合并得到。
- `07_5_model_coverage.csv` 用于检查哪些 `07_4` 模型还没有生成对应 MUSDB 输出。
